# `groundinsight` — reduction factor & grounding impedance

Plausibility tests on the two physically derived result quantities:

- **Reduction factor** `r = |u_with_mutual| / |u_without_mutual|` at the
  fault bus.
- **Grounding impedance** `Z_G = u_EPR / (r * I_fault)` at the fault
  bus, returned per frequency.

**Checks**
1. Removing the shield (OHL only) collapses `r` to 1 and `Z_G` to the
   pure parallel combination of bus impedances along the line.
2. With cable impedances `r` matches the analytical value
   `|1 - Z_m/Z_s|`.
3. Sweep over `rho` shows the expected monotonic Z_G trend on a soil
   resistivity-dependent bus formula (`Z_bus = rho/100`).


In [ ]:
import sys
import os
# Make the src/ tree importable when running from the notebooks/ folder
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import groundinsight as gi
from groundinsight.models.core_models import BusType, BranchType, ComplexNumber

print('groundinsight', gi.__version__)


In [ ]:
# Reference impedances (see tests/test_topology_and_reduction.py).
# With these values the analytical reduction factor for a single MV cable is
#     r = |1 - Z_mutual / Z_self| = 0.3846...
# which sits inside the field reference band of 0.30 ... 0.40.
Z_SELF   = complex(0.25, 0.6)
Z_MUTUAL = complex(0.0,  0.6)
R_REF    = abs(1.0 - Z_MUTUAL / Z_SELF)
print(f'analytical r = {R_REF:.4f}')


In [ ]:
def make_bus_type():
    """Unit bus impedance so the shield path dominates."""
    return BusType(
        name='BusUnit',
        description='Unit-like bus impedance for plausibility tests',
        system_type='Grounded',
        voltage_level=20.0,
        impedance_formula='rho * 0 + 1.0 + I * f * 0',
    )

def make_ms_cable():
    """MV cable with the reference impedances above."""
    return BranchType(
        name='MSCable',
        description='MV cable reference branch',
        grounding_conductor=True,
        self_impedance_formula='(rho * 0 + 0.25 + I * 0.6)*l',
        mutual_impedance_formula='(rho * 0 + 0.0 + I * 0.6)*l',
    )

def make_ohl():
    """Overhead line without shield."""
    return BranchType(
        name='OHLine',
        description='Overhead line without shield',
        grounding_conductor=False,
        self_impedance_formula='NaN',
        mutual_impedance_formula='NaN',
    )


## 1. Limit case: no shield -> r = 1

In [ ]:
def two_bus_only(branch_type, name='OHLOnly'):
    net = gi.create_network(name=name, frequencies=[50])
    bus_type = make_bus_type()
    gi.create_bus(name='bus1', type=bus_type, network=net)
    gi.create_bus(name='bus2', type=bus_type, network=net)
    gi.create_branch(name='b12', type=branch_type, from_bus='bus1',
                     to_bus='bus2', length=1.0, network=net)
    gi.create_source(name='src',   bus='bus1', values={50: 100.0}, network=net)
    gi.create_fault (name='fault', bus='bus2', scalings={50: 1.0},  network=net)
    return net

net_ohl   = two_bus_only(make_ohl(),       name='OHLOnly')
net_cable = two_bus_only(make_ms_cable(),  name='CableOnly')

gi.run_fault(net_ohl,   fault_name='fault')
gi.run_fault(net_cable, fault_name='fault')

r_ohl   = net_ohl.results['fault'].reduction_factor.value[50.0]
r_cable = net_cable.results['fault'].reduction_factor.value[50.0]

print(f'OHL only   r = {r_ohl:.4f}    (must be 1)')
print(f'cable only r = {r_cable:.4f}  (must be {R_REF:.4f})')
assert abs(r_ohl - 1.0) < 1e-9
assert abs(r_cable - R_REF) < 1e-3


## 2. Frequency dependence

Re-run the cable-only case across several frequencies. Because the
reference branch has `Z_m = j*0.6` and `Z_s = 0.25 + j*0.6` *constant*
(no f-dependence in the formula here), `r` should be flat in f.
This is a useful sanity check: anything that varies with frequency
here would point to an unintended frequency hook.


In [ ]:
freqs = [50.0, 100.0, 250.0, 500.0, 1000.0]
net_freq = gi.create_network(name='CableSweepF', frequencies=freqs)
bus_type = make_bus_type()
cable = make_ms_cable()
gi.create_bus(name='bus1', type=bus_type, network=net_freq)
gi.create_bus(name='bus2', type=bus_type, network=net_freq)
gi.create_branch(name='b12', type=cable, from_bus='bus1', to_bus='bus2',
                 length=1.0, network=net_freq)
gi.create_source(name='src',   bus='bus1',
                 values={f: 100.0 for f in freqs}, network=net_freq)
gi.create_fault (name='fault', bus='bus2',
                 scalings={f: 1.0 for f in freqs}, network=net_freq)

gi.run_fault(net_freq, fault_name='fault')
rf = net_freq.results['fault'].reduction_factor.value
for f in freqs:
    print(f'  f = {f:6.1f} Hz   r = {rf[f]:.6f}')
flat = all(abs(rf[f] - R_REF) < 1e-3 for f in freqs)
print(f'r flat across frequencies: {flat}')
assert flat


## 3. Soil-resistivity sweep on Z_G

Use a `rho`-dependent bus impedance (`impedance_formula = 'rho/100'`)
so the sweep actually moves Z_G. The cable impedance stays
`rho`-independent here.

Expectation: Z_G grows monotonically with `rho`.


In [ ]:
bus_type_rho = BusType(
    name='RhoBus',
    description='Bus impedance proportional to rho',
    system_type='Grounded',
    voltage_level=20.0,
    impedance_formula='rho / 100',
)

def two_bus_rho(rho):
    net = gi.create_network(name=f'rho{rho}', frequencies=[50])
    cable = make_ms_cable()
    gi.create_bus(name='bus1', type=bus_type_rho,
                  specific_earth_resistance=rho, network=net)
    gi.create_bus(name='bus2', type=bus_type_rho,
                  specific_earth_resistance=rho, network=net)
    gi.create_branch(name='b12', type=cable, from_bus='bus1', to_bus='bus2',
                     length=1.0, specific_earth_resistance=rho, network=net)
    gi.create_source(name='src',   bus='bus1', values={50: 100.0}, network=net)
    gi.create_fault (name='fault', bus='bus2', scalings={50: 1.0},  network=net)
    return net

rhos = [50.0, 100.0, 200.0, 500.0, 1000.0]
ZG = []
for rho in rhos:
    n = two_bus_rho(rho)
    gi.run_fault(n, fault_name='fault')
    z = n.results['fault'].grounding_impedance.value[50.0]
    ZG.append(abs(complex(z.real, z.imag)))
    print(f'  rho = {rho:6.0f} Ohm.m  |Z_G| = {ZG[-1]:7.4f} Ohm')

monotonic = all(ZG[i+1] >= ZG[i] for i in range(len(ZG)-1))
print(f'Z_G monotonic in rho: {monotonic}')
assert monotonic


## 4. Plot Z_G(rho)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rhos, ZG, 'o-')
ax.set_xlabel('soil resistivity rho in Ohm.m')
ax.set_ylabel('|Z_G| in Ohm at 50 Hz')
ax.set_title('Z_G vs. rho for a 2-bus MV cable link')
ax.grid(alpha=0.3)
plt.show()


---

**Modelling assumptions**

- `r` is computed from two solves: with and without the mutual Norton
  injection. It is a magnitude ratio at the fault bus, not a phasor
  ratio.
- `Z_G = u_EPR / (r * I_fault)` divides by `r`, so `r = 0` returns
  `None`. The far-end-of-an-OHL case never triggers this in practice
  because `r = 1` there.
- The bus formula `rho/100` is purely a sweep handle here. Real bus
  impedances are typically given by Sunde / mesh / ring formulas
  (see `notebooks/05_plotting.ipynb` and `docs/examples/`).
